# Yearly Values Code

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Load full discharge dataset
input_path = '/content/drive/My Drive/Colorado_Discharge_1980-2024.csv'
df = pd.read_csv(input_path)
df.columns = df.columns.str.strip().str.lower()

# Clean and convert
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['cubic_ft_sec'] = pd.to_numeric(df['cubic_ft_sec'], errors='coerce')
df.dropna(subset=['year', 'cubic_ft_sec'], inplace=True)

# Group by year and calculate mean discharge
yearly_avg = df.groupby('year', as_index=False)['cubic_ft_sec'].mean()
yearly_avg.rename(columns={'cubic_ft_sec': 'avg_discharge_cfs'}, inplace=True)

# Preview output
print(yearly_avg.head())

# Save to Google Drive
output_path = '/content/drive/My Drive/colorado_discharge_yearly_average.csv'
yearly_avg.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   year  avg_discharge_cfs
0  1980        1143.844444
1  1981         561.011111
2  1982         859.960000
3  1983        1352.810000
4  1984        1581.109091
Saved to: /content/drive/My Drive/colorado_discharge_yearly_average.csv


In [ ]:
!pip install mpld3
import mpld3
# Please restart the runtime after installing the library.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 3.6 MB/s eta 0:00:00


# Linear Regression

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive')

# Load and clean SWE dataset
file_path = '/content/drive/My Drive/yearly_swe_average.csv'
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip().str.lower()
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['swe_average'] = pd.to_numeric(df['swe_average'], errors='coerce')
df.dropna(subset=['year', 'swe_average'], inplace=True)

# Segment periods
df['period'] = np.where(df['year'] <= 2001, 'SWE 1980–2001', 'SWE 2002–2024')
df_1980_2001 = df[df['period'] == 'SWE 1980–2001']
df_2002_2024 = df[df['period'] == 'SWE 2002–2024']

# Regression line and means
slope, intercept = np.polyfit(df['year'], df['swe_average'], 1)
df['regression_line'] = slope * df['year'] + intercept
mean_1980_2001 = df_1980_2001['swe_average'].mean()
mean_2002_2024 = df_2002_2024['swe_average'].mean()

# Build Plotly figure
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_1980_2001['year'], y=df_1980_2001['swe_average'],
    mode='markers', name='SWE 1980–2001',
    marker=dict(color='blue', size=7),
    hovertemplate='Year: %{x}<br>SWE: %{y:.2f} in<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=df_2002_2024['year'], y=df_2002_2024['swe_average'],
    mode='markers', name='SWE 2002–2024',
    marker=dict(color='green', size=7),
    hovertemplate='Year: %{x}<br>SWE: %{y:.2f} in<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=df['year'], y=df['regression_line'],
    mode='lines', name='Linear Trend',
    line=dict(color='red', dash='dash'),
    hoverinfo='skip'
))
fig.add_shape(type='line', x0=1980, x1=2001,
    y0=mean_1980_2001, y1=mean_1980_2001,
    line=dict(color='blue', dash='dot'))
fig.add_shape(type='line', x0=2002, x1=2024,
    y0=mean_2002_2024, y1=mean_2002_2024,
    line=dict(color='green', dash='dot'))

fig.update_layout(
    title='Snowpack Regression Colorado (Segmented)',
    xaxis_title='Year',
    yaxis_title='Snow Water Equivalency (SWE) in.',
    hovermode='x',
    template='plotly_white',
    font=dict(family='Segoe UI', size=14),
    title_font=dict(size=20),
    legend=dict(font=dict(size=14)),
    autosize=True
)

# Generate chart HTML
html_chart = fig.to_html(full_html=False, include_plotlyjs='cdn')

# Build final HTML with proper layout
html_summary = f"""
<div id="summary-block">
  <h3>Regression Summary</h3>
  <p><strong>Trend Slope:</strong> {slope:.4f} in/year</p>
  <p><strong>Trend Intercept:</strong> {intercept:.2f} in</p>
  <p><strong>Mean SWE (1980–2001):</strong> {mean_1980_2001:.2f} in</p>
  <p><strong>Mean SWE (2002–2024):</strong> {mean_2002_2024:.2f} in</p>
</div>
"""

full_html = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <title>SWE Regression Report</title>
  <style>
    html, body {{
      margin: 0;
      padding: 0;
      height: 100%;
      background-color: #f9f9fb;
      font-family: 'Segoe UI', sans-serif;
      display: flex;
      flex-direction: column;
    }}
    #chart-container {{
      flex: 0 0 87.5%;
      width: 100%;
      height: 87.5%;
      box-sizing: border-box;
      overflow: hidden;
    }}
    #chart-container > div {{
      width: 100% !important;
      height: 100% !important;
    }}
    #summary-block {{
      flex: 0 0 12.5%;
      background-color: #f3f5f9;
      font-size: 14px;
      padding: 12px 25px;
      text-align: center;
      border-top: 1px solid #ddd;
      box-shadow: 0 -1px 3px rgba(0,0,0,0.05);
    }}
    #summary-block h3 {{
      margin: 6px 0;
      font-size: 16px;
      color: #2c3e50;
    }}
    #summary-block p {{
      margin: 3px 0;
      line-height: 1.4;
    }}
  </style>
</head>
<body>
  <div id="chart-container">
    {html_chart}
  </div>
  {html_summary}
</body>
</html>
"""

# Save and download
html_path = '/content/swe_regression_fullscreen_fixed.html'
with open(html_path, 'w') as f:
    f.write(full_html)
files.download(html_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive')

# Load averaged discharge data
file_path = '/content/drive/My Drive/colorado_discharge_yearly_average.csv'
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip().str.lower()

# Clean and segment
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['avg_discharge_cfs'] = pd.to_numeric(df['avg_discharge_cfs'], errors='coerce')
df.dropna(subset=['year', 'avg_discharge_cfs'], inplace=True)

df['period'] = np.where(df['year'] <= 2001, '1980–2001', '2002–2024')
df_early = df[df['period'] == '1980–2001']
df_late = df[df['period'] == '2002–2024']

# Regression line + means
slope, intercept = np.polyfit(df['year'], df['avg_discharge_cfs'], 1)
df['regression_line'] = slope * df['year'] + intercept
mean_early = df_early['avg_discharge_cfs'].mean()
mean_late = df_late['avg_discharge_cfs'].mean()

# Build interactive figure
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_early['year'], y=df_early['avg_discharge_cfs'],
    mode='markers', name='1980–2001',
    marker=dict(color='blue', size=7),
    hovertemplate='Year: %{x}<br>Discharge: %{y:.1f} cfs<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=df_late['year'], y=df_late['avg_discharge_cfs'],
    mode='markers', name='2002–2024',
    marker=dict(color='green', size=7),
    hovertemplate='Year: %{x}<br>Discharge: %{y:.1f} cfs<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=df['year'], y=df['regression_line'],
    mode='lines', name='Linear Trend',
    line=dict(color='red', dash='dash'),
    hoverinfo='skip'
))
fig.add_shape(type='line', x0=1980, x1=2001,
    y0=mean_early, y1=mean_early,
    line=dict(color='blue', dash='dot'))
fig.add_shape(type='line', x0=2002, x1=2024,
    y0=mean_late, y1=mean_late,
    line=dict(color='green', dash='dot'))

fig.update_layout(
    title='Statewide Colorado Discharge Regression (1980–2024)',
    xaxis_title='Year',
    yaxis_title='Average Streamflow (cfs)',
    hovermode='x',
    template='plotly_white',
    font=dict(family='Segoe UI', size=14),
    title_font=dict(size=20),
    legend=dict(font=dict(size=14)),
    autosize=True
)

# Export full HTML
chart_html = fig.to_html(full_html=False, include_plotlyjs='cdn')
summary_html = f"""
<div id="summary-block">
  <h3>Regression Summary</h3>
  <p><strong>Trend Slope:</strong> {slope:.4f} cfs/year</p>
  <p><strong>Trend Intercept:</strong> {intercept:.2f} cfs</p>
  <p><strong>Mean Discharge (1980–2001):</strong> {mean_early:.1f} cfs</p>
  <p><strong>Mean Discharge (2002–2024):</strong> {mean_late:.1f} cfs</p>
</div>
"""

full_html = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <title>Discharge Regression Report</title>
  <style>
    html, body {{
      margin: 0;
      padding: 0;
      height: 100%;
      background-color: #f9f9fb;
      font-family: 'Segoe UI', sans-serif;
      display: flex;
      flex-direction: column;
    }}
    #chart-container {{
      flex: 0 0 87.5%;
      width: 100%;
      height: 87.5%;
      box-sizing: border-box;
      overflow: hidden;
    }}
    #chart-container > div {{
      width: 100% !important;
      height: 100% !important;
    }}
    #summary-block {{
      flex: 0 0 12.5%;
      background-color: #f3f5f9;
      font-size: 14px;
      padding: 12px 25px;
      text-align: center;
      border-top: 1px solid #ddd;
      box-shadow: 0 -1px 3px rgba(0,0,0,0.05);
    }}
    #summary-block h3 {{
      margin: 6px 0;
      font-size: 16px;
      color: #2c3e50;
    }}
    #summary-block p {{
      margin: 3px 0;
      line-height: 1.4;
    }}
  </style>
</head>
<body>
  <div id="chart-container">
    {chart_html}
  </div>
  {summary_html}
</body>
</html>
"""

# Save to Drive and download
output_path = '/content/colorado_discharge_regression_plot.html'
with open(output_path, 'w') as f:
    f.write(full_html)
files.download(output_path)

Mounted at /content/drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Harmonic Plots

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from google.colab import drive, files

# Mount Drive
drive.mount('/content/drive')

# Load data
file_path = '/content/drive/My Drive/yearly_swe_average.csv'
df = pd.read_csv(file_path)
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['swe_average'] = pd.to_numeric(df['swe_average'], errors='coerce')
df.dropna(subset=['year', 'swe_average'], inplace=True)

# Segment and normalize
df1 = df[(df['year'] >= 1980) & (df['year'] <= 2001)]
df2 = df[(df['year'] >= 2002) & (df['year'] <= 2024)]
years1, swe1 = df1['year'].values, df1['swe_average'].values
years2, swe2 = df2['year'].values, df2['swe_average'].values
mean1, mean2 = np.mean(swe1), np.mean(swe2)

t1 = 2 * np.pi * (years1 - years1.min()) / (years1.max() - years1.min())
t2 = 2 * np.pi * (years2 - years2.min()) / (years2.max() - years2.min())

# Harmonic model
def multi_harmonic(t, *params):
    n = (len(params) - 1) // 2
    result = np.zeros_like(t)
    for i in range(n):
        A = params[2*i]
        B = params[2*i + 1]
        result += A * np.sin((i+1)*t) + B * np.cos((i+1)*t)
    return result + params[-1]

# Fit models
n_harmonics = 10
p0_1 = [1] * (2 * n_harmonics) + [mean1]
popt1, _ = curve_fit(multi_harmonic, t1, swe1, p0=p0_1, maxfev=10000)
fit1 = multi_harmonic(t1, *popt1)

p0_2 = [1] * (2 * n_harmonics) + [mean2]
popt2, _ = curve_fit(multi_harmonic, t2, swe2, p0=p0_2, maxfev=10000)
fit2 = multi_harmonic(t2, *popt2)

# Build Plotly figure
fig = go.Figure()
fig.add_trace(go.Scatter(x=years1, y=swe1,
    mode='markers', name='SWE 1980–2001',
    marker=dict(color='blue', size=6),
    hovertemplate='Year: %{x}<br>SWE: %{y:.2f} in<extra></extra>'))
fig.add_trace(go.Scatter(x=years2, y=swe2,
    mode='markers', name='SWE 2002–2024',
    marker=dict(color='green', size=6),
    hovertemplate='Year: %{x}<br>SWE: %{y:.2f} in<extra></extra>'))
fig.add_trace(go.Scatter(x=years1, y=fit1,
    mode='lines', name='Harmonic Fit 1980–2001',
    line=dict(color='navy', dash='dash'),
    hoverinfo='skip'))
fig.add_trace(go.Scatter(x=years2, y=fit2,
    mode='lines', name='Harmonic Fit 2002–2024',
    line=dict(color='darkgreen', dash='dash'),
    hoverinfo='skip'))

fig.update_layout(
    title='Enhanced Harmonic Snowpack Trends (1980–2024)',
    xaxis_title='Year',
    yaxis_title='Snow Water Equivalency (SWE) in.',
    hovermode='x',
    template='plotly_white',
    font=dict(family='Segoe UI', size=14),
    title_font=dict(size=20),
    legend=dict(font=dict(size=14)),
    autosize=True,
    height=750
)

# Assemble full HTML
chart_html = fig.to_html(full_html=False, include_plotlyjs='cdn')
summary_html = f"""
<div id="summary-block">
  <h3>Period Mean SWE</h3>
  <p><strong>1980–2001:</strong> {mean1:.2f} in</p>
  <p><strong>2002–2024:</strong> {mean2:.2f} in</p>
</div>
"""

full_html = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <title>Harmonic SWE Plot</title>
  <style>
    html, body {{
      margin: 0;
      padding: 0;
      height: 100%;
      font-family: 'Segoe UI', sans-serif;
      display: flex;
      flex-direction: column;
      background-color: #f9f9fb;
    }}
    #plot-section {{
      flex: 0 0 87.5%;
      width: 100%;
      height: 87.5%;
      box-sizing: border-box;
    }}
    #plot-section > div {{
      width: 100% !important;
      height: 100% !important;
    }}
    #summary-block {{
      flex: 0 0 12.5%;
      background-color: #f3f5f9;
      text-align: center;
      font-size: 14px;
      padding: 12px 25px;
      border-top: 1px solid #ddd;
      box-shadow: 0 -1px 3px rgba(0,0,0,0.05);
    }}
    #summary-block h3 {{
      margin: 6px 0;
      font-size: 16px;
      color: #2c3e50;
    }}
    #summary-block p {{
      margin: 4px 0;
    }}
  </style>
</head>
<body>
  <div id="plot-section">
    {chart_html}
  </div>
  {summary_html}
</body>
</html>
"""

# Export
html_path = "/content/swe_harmonic_vertical_hover.html"
with open(html_path, "w") as f:
    f.write(full_html)

files.download(html_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from google.colab import drive, files

# Mount Drive
drive.mount('/content/drive')

# Load discharge dataset
file_path = '/content/drive/My Drive/colorado_discharge_yearly_average.csv'
df = pd.read_csv(file_path)
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['avg_discharge_cfs'] = pd.to_numeric(df['avg_discharge_cfs'], errors='coerce')
df.dropna(subset=['year', 'avg_discharge_cfs'], inplace=True)

# Segment and normalize
df1 = df[(df['year'] >= 1980) & (df['year'] <= 2001)]
df2 = df[(df['year'] >= 2002) & (df['year'] <= 2024)]
years1, flow1 = df1['year'].values, df1['avg_discharge_cfs'].values
years2, flow2 = df2['year'].values, df2['avg_discharge_cfs'].values
mean1, mean2 = np.mean(flow1), np.mean(flow2)

t1 = 2 * np.pi * (years1 - years1.min()) / (years1.max() - years1.min())
t2 = 2 * np.pi * (years2 - years2.min()) / (years2.max() - years2.min())

# Harmonic model
def multi_harmonic(t, *params):
    n = (len(params) - 1) // 2
    result = np.zeros_like(t)
    for i in range(n):
        A = params[2*i]
        B = params[2*i + 1]
        result += A * np.sin((i+1)*t) + B * np.cos((i+1)*t)
    return result + params[-1]

# Fit models
n_harmonics = 10
p0_1 = [1] * (2 * n_harmonics) + [mean1]
popt1, _ = curve_fit(multi_harmonic, t1, flow1, p0=p0_1, maxfev=10000)
fit1 = multi_harmonic(t1, *popt1)

p0_2 = [1] * (2 * n_harmonics) + [mean2]
popt2, _ = curve_fit(multi_harmonic, t2, flow2, p0=p0_2, maxfev=10000)
fit2 = multi_harmonic(t2, *popt2)

# Build Plotly figure
fig = go.Figure()
fig.add_trace(go.Scatter(x=years1, y=flow1,
    mode='markers', name='Discharge 1980–2001',
    marker=dict(color='blue', size=6),
    hovertemplate='Year: %{x}<br>Discharge: %{y:.1f} cfs<extra></extra>'))
fig.add_trace(go.Scatter(x=years2, y=flow2,
    mode='markers', name='Discharge 2002–2024',
    marker=dict(color='green', size=6),
    hovertemplate='Year: %{x}<br>Discharge: %{y:.1f} cfs<extra></extra>'))
fig.add_trace(go.Scatter(x=years1, y=fit1,
    mode='lines', name='Harmonic Fit 1980–2001',
    line=dict(color='navy', dash='dash'),
    hoverinfo='skip'))
fig.add_trace(go.Scatter(x=years2, y=fit2,
    mode='lines', name='Harmonic Fit 2002–2024',
    line=dict(color='darkgreen', dash='dash'),
    hoverinfo='skip'))

fig.update_layout(
    title='Enhanced Harmonic Discharge Trends (1980–2024)',
    xaxis_title='Year',
    yaxis_title='Average Streamflow (cfs)',
    hovermode='x',
    template='plotly_white',
    font=dict(family='Segoe UI', size=14),
    title_font=dict(size=20),
    legend=dict(font=dict(size=14)),
    autosize=True,
    height=750
)

# Assemble full HTML
chart_html = fig.to_html(full_html=False, include_plotlyjs='cdn')
summary_html = f"""
<div id="summary-block">
  <h3>Period Mean Discharge</h3>
  <p><strong>1980–2001:</strong> {mean1:.1f} cfs</p>
  <p><strong>2002–2024:</strong> {mean2:.1f} cfs</p>
</div>
"""

full_html = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <title>Harmonic Discharge Plot</title>
  <style>
    html, body {{
      margin: 0;
      padding: 0;
      height: 100%;
      font-family: 'Segoe UI', sans-serif;
      display: flex;
      flex-direction: column;
      background-color: #f9f9fb;
    }}
    #plot-section {{
      flex: 0 0 87.5%;
      width: 100%;
      height: 87.5%;
      box-sizing: border-box;
    }}
    #plot-section > div {{
      width: 100% !important;
      height: 100% !important;
    }}
    #summary-block {{
      flex: 0 0 12.5%;
      background-color: #f3f5f9;
      text-align: center;
      font-size: 14px;
      padding: 12px 25px;
      border-top: 1px solid #ddd;
      box-shadow: 0 -1px 3px rgba(0,0,0,0.05);
    }}
    #summary-block h3 {{
      margin: 6px 0;
      font-size: 16px;
      color: #2c3e50;
    }}
    #summary-block p {{
      margin: 4px 0;
    }}
  </style>
</head>
<body>
  <div id="plot-section">
    {chart_html}
  </div>
  {summary_html}
</body>
</html>
"""

# Save and download
html_path = "/content/colorado_discharge_harmonic_plot.html"
with open(html_path, "w") as f:
    f.write(full_html)

files.download(html_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Combined Map With Point and Polygon Data

In [ ]:
# prompt: Awesome, I have the 1980.geojson file, 2002.geojson file, dis_avg_1980p.geojson and dis_avg_2002p.geojson files uploaded. For the 1980 and 2002, use the correct color scaling empazing more color distribtions in lesser swe values to best demonstrate the differentces. For the dis_avg geojson files, since they are point data, add emphasis on the point values based off the cubic feet per second. The biger the values, the bigger the point along with appropiate color coding. ENsure each point or polygon includes a hover feature that displays their features. Make sure the output is in html file that automatically downloads. Esure each geojson file has its own layer. Incude a select all and deselect all function please.

import folium
from folium.plugins import Fullscreen, MousePosition, MeasureControl
from folium.utilities import JsCode
import geopandas as gpd # Import geopandas
import pandas as pd # Import pandas
from google.colab import drive # Import drive
from google.colab import files # Import files
import os # Import os
from branca.colormap import linear, StepColormap # Import colormap tools


# ✅ Step 1: Install necessary packages if not already done by previous cells
# !pip install geemap folium geopandas pandas

# ✅ Step 2: Import libraries (already mostly done above, but ensure Folium is imported)
# import folium
# import geopandas as gpd
# import pandas as pd
# from google.colab import drive
# from google.colab import files
# import os # Already imported

# ✅ Step 3: Mount Google Drive (Already done in previous cells)
drive.mount('/content/drive')

# Define file paths for the GeoJSON files
geojson_swe_1980_path = '/content/drive/My Drive/1980.geojson'  # Assuming this is the correct path for SWE 1980
geojson_swe_2002_path = '/content/drive/My Drive/2002.geojson'  # Assuming this is the correct path for SWE 2002
geojson_dis_avg_1980_path = '/content/drive/My Drive/dis_avg_1980p.geojson' # Assuming this is the correct path for Discharge 1980 avg
geojson_dis_avg_2002_path = '/content/drive/My Drive/dis_avg_2002p.geojson' # Assuming this is the correct path for Discharge 2002 avg


# Check if files exist before proceeding
file_paths_to_check = [geojson_swe_1980_path, geojson_swe_2002_path, geojson_dis_avg_1980_path, geojson_dis_avg_2002_path]
for fp in file_paths_to_check:
    if not os.path.exists(fp):
        print(f"Error: File not found - {fp}. Please ensure the GeoJSON files are uploaded to your Google Drive.")
        # Consider adding a break or raising an error here if files are missing
        # break # Or raise FileNotFoundError(f"Missing file: {fp}")


# ✅ Step 4: Create a base map
# Use a center and zoom level appropriate for Colorado
m = folium.Map(location=[39.0, -105.5], zoom_start=6, control_scale=True)

# Add plugins
Fullscreen().add_to(m)
MousePosition().add_to(m)
MeasureControl().add_to(m)


# ✅ Step 5: Add Polygon Layers (1980 and 2002 SWE GeoJSON)
try:
    if os.path.exists(geojson_swe_1980_path) and os.path.exists(geojson_swe_2002_path):
        # Load GeoJSON files into GeoDataFrames
        gdf_1980_swe = gpd.read_file(geojson_swe_1980_path)
        gdf_2002_swe = gpd.read_file(geojson_swe_2002_path)

         # Ensure GeoDataFrames are in WGS84 (EPSG:4326)
        if gdf_1980_swe.crs != "EPSG:4326":
            gdf_1980_swe = gdf_1980_swe.to_crs("EPSG:4326")
        if gdf_2002_swe.crs != "EPSG:4326":
            gdf_2002_swe = gdf_2002_swe.to_crs("EPSG:4326")


        # Define the column containing the SWE values
        swe_column = 'Snow_Water_Equivalent__in__Mean_of_Monthly_Values'

        # Check if the SWE column exists in both GeoDataFrames
        if swe_column not in gdf_1980_swe.columns:
            print(f"Error: SWE column '{swe_column}' not found in {geojson_swe_1980_path}. Found columns: {gdf_1980_swe.columns.tolist()}")
        if swe_column not in gdf_2002_swe.columns:
             print(f"Error: SWE column '{swe_column}' not found in {geojson_swe_2002_path}. Found columns: {gdf_2002_swe.columns.tolist()}")

        if swe_column in gdf_1980_swe.columns and swe_column in gdf_2002_swe.columns:

            # Determine a consistent color scale based on the range of SWE values in both datasets
            all_swe_values = pd.concat([gdf_1980_swe[swe_column], gdf_2002_swe[swe_column]])
            min_swe = all_swe_values.min()
            max_swe = all_swe_values.max()

            # Define color scales emphasizing lower SWE values using quantiles
            # You can adjust the number of bins (k) and the quantiles as needed
            swe_quantiles = pd.concat([gdf_1980_swe[swe_column], gdf_2002_swe[swe_column]]).quantile([0, 0.1, 0.3, 0.5, 0.7, 0.9, 1]).tolist()

            # Define a color map (e.g., Blues or YlGnBu for lower values emphasis) - Using a darker palette
            swe_colormap = StepColormap(['#ffffcc','#c7e9b4','#7fcdbb','#41b6c4','#1d91c0','#225ea8','#0c2c84'],
                                        index=swe_quantiles,
                                        vmin=min(swe_quantiles),
                                        vmax=max(swe_quantiles))
            swe_colormap.caption = 'Snow Water Equivalency (in.)'


            # Add 1980 SWE Layer
            folium.GeoJson(
                gdf_1980_swe.to_json(),
                name='SWE 1980',
                style_function=lambda feature: {
                    'fillColor': swe_colormap(feature['properties'].get(swe_column, 0)),
                    'color': 'white',
                    'weight': 1,
                    'fillOpacity': 0.7,
                },
                tooltip=folium.GeoJsonTooltip(
                    fields=[swe_column],
                    aliases=['SWE Average:'],
                    localize=True,
                    sticky=False,
                    labels=True,
                    max_width=800,
                    style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 6px;")
                )
            ).add_to(m)


            # Add 2002 SWE Layer
            folium.GeoJson(
                gdf_2002_swe.to_json(),
                name='SWE 2002',
                style_function=lambda feature: {
                    'fillColor': swe_colormap(feature['properties'].get(swe_column, 0)),
                    'color': 'white',
                    'weight': 1,
                    'fillOpacity': 0.7,
                },
                tooltip=folium.GeoJsonTooltip(
                    fields=[swe_column],
                    aliases=['SWE Average:'],
                    localize=True,
                    sticky=False,
                    labels=True,
                    max_width=800,
                    style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 6px;")
                )
            ).add_to(m)

            # Add SWE color map legend
            swe_colormap.add_to(m)


        else:
            print("Skipping SWE polygon layers due to missing SWE column.")


except FileNotFoundError as e:
    print(f"Skipping SWE polygon layers: {e}")
except Exception as e:
    print(f"An error occurred while adding SWE polygon layers: {e}")


# ✅ Step 6: Add Point Layers (Discharge GeoJSON)
try:
    if os.path.exists(geojson_dis_avg_1980_path) and os.path.exists(geojson_dis_avg_2002_path):
        # Load GeoJSON files into GeoDataFrames
        gdf_dis_1980 = gpd.read_file(geojson_dis_avg_1980_path)
        gdf_dis_2002 = gpd.read_file(geojson_dis_avg_2002_path)

         # Ensure GeoDataFrames are in WGS84 (EPSG:4326)
        if gdf_dis_1980.crs != "EPSG:4326":
            gdf_dis_1980 = gdf_dis_1980.to_crs("EPSG:4326")
        if gdf_dis_2002.crs != "EPSG:4326":
            gdf_dis_2002 = gdf_dis_2002.to_crs("EPSG:4326")


        # Ensure the cubic feet per second column exists and is numeric
        discharge_col_1980 = 'average_cubic_ft_sec_1980_2001'
        discharge_col_2002 = 'average_cubic_ft_sec_2002_2024'

        if discharge_col_1980 not in gdf_dis_1980.columns:
            print(f"Error: Discharge column '{discharge_col_1980}' not found in {geojson_dis_avg_1980_path}. Found columns: {gdf_dis_1980.columns.tolist()}")
        if discharge_col_2002 not in gdf_dis_2002.columns:
             print(f"Error: Discharge column '{discharge_col_2002}' not found in {geojson_dis_avg_2002_path}. Found columns: {gdf_dis_2002.columns.tolist()}")


        if discharge_col_1980 in gdf_dis_1980.columns and discharge_col_2002 in gdf_dis_2002.columns:

            gdf_dis_1980[discharge_col_1980] = pd.to_numeric(gdf_dis_1980[discharge_col_1980], errors='coerce')
            gdf_dis_2002[discharge_col_2002] = pd.to_numeric(gdf_dis_2002[discharge_col_2002], errors='coerce')

            # Drop rows with NaN discharge values
            gdf_dis_1980.dropna(subset=[discharge_col_1980], inplace=True)
            gdf_dis_2002.dropna(subset=[discharge_col_2002], inplace=True)

            # Determine min/max values across both datasets for consistent scaling
            min_discharge = min(gdf_dis_1980[discharge_col_1980].min(), gdf_dis_2002[discharge_col_2002].min())
            max_discharge = max(gdf_dis_1980[discharge_col_1980].max(), gdf_dis_2002[discharge_col_2002].max())

            # Define a color scale for discharge (using quantiles to emphasize lower values) - Using a darker palette
            discharge_quantiles = pd.concat([gdf_dis_1980[discharge_col_1980], gdf_dis_2002[discharge_col_2002]]).quantile([0, 0.25, 0.5, 0.75, 1.0]).tolist()
            discharge_colors = ['#fee5d9','#fcae91','#fb6a4a','#de2d26','#a50f15'] # Example: Shades of red
            discharge_colormap = StepColormap(discharge_colors,
                                            index=discharge_quantiles,
                                            vmin=min(discharge_quantiles),
                                            vmax=max(discharge_quantiles))
            discharge_colormap.caption = 'Average Discharge (cubic ft/sec)'


            # Define a size scale for points based on discharge
            # Use a linear scale for size
            min_radius = 6 # Increased minimum radius
            max_radius = 20 # Increased maximum radius

            def get_radius(value, min_val, max_val, min_r, max_r):
                if max_val == min_val: return min_r # Avoid division by zero
                # Linear scaling
                return min_r + (max_r - min_r) * (value - min_val) / (max_val - min_val)

            # Add 1980 Discharge Layer
            # Use a FeatureGroup to add markers and tooltips within one layer control entry
            fg_dis_1980 = folium.FeatureGroup(name='Discharge 1980 (avg cfs)')

            for _, row in gdf_dis_1980.iterrows():
                discharge_val = row[discharge_col_1980]
                radius = get_radius(discharge_val, min_discharge, max_discharge, min_radius, max_radius)

                # Define point color based on value using the discharge colormap
                color = discharge_colormap(discharge_val)


                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    radius=radius,
                    color=color,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.7,
                    tooltip=f"Location: {row.get('location', 'N/A')}<br>Avg Discharge: {discharge_val:.2f} cfs" # Customize tooltip fields
                ).add_to(fg_dis_1980)

            fg_dis_1980.add_to(m)


            # Add 2002 Discharge Layer
            fg_dis_2002 = folium.FeatureGroup(name='Discharge 2002 (avg cfs)')

            for _, row in gdf_dis_2002.iterrows():
                discharge_val = row[discharge_col_2002]
                radius = get_radius(discharge_val, min_discharge, max_discharge, min_radius, max_radius)

                # Define point color based on value using the discharge colormap
                color = discharge_colormap(discharge_val)


                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    radius=radius,
                    color=color,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.7,
                    tooltip=f"Location: {row.get('location', 'N/A')}<br>Avg Discharge: {discharge_val:.2f} cfs" # Customize tooltip fields
                ).add_to(fg_dis_2002)

            fg_dis_2002.add_to(m)

            # Add Discharge color map legend
            discharge_colormap.add_to(m)

        else:
            print("Skipping discharge point layers due to missing discharge column.")


except FileNotFoundError as e:
    print(f"Skipping discharge point layers: {e}")
except Exception as e:
    print(f"An error occurred while adding discharge point layers: {e}")


# ✅ Step 7: Add Layer Control with Select All/Deselect All
# Add standard Layer Control first to see layers
folium.LayerControl().add_to(m)

# Remove custom buttons for Select All/Deselect All
# (No longer adding the HTML/JS for these buttons)


# ✅ Step 8: Save the map as an HTML file and trigger download
output_html_path = 'colorado_combined_map.html'
m.save(output_html_path)

print(f"Interactive map saved to {output_html_path}")

# Trigger download
try:
    files.download(output_html_path)
    print(f"File '{output_html_path}' downloaded.")
except FileNotFoundError:
    print(f"Error: Could not find the generated HTML file at {output_html_path} for download.")
except Exception as e:
    print(f"An error occurred during file download: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Interactive map saved to colorado_combined_map.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File 'colorado_combined_map.html' downloaded.
